# Multi-Rat Sliding-Window Decoding Curve Validation (All 5 Rats)

**Purpose:** notebook 008 found that a later window (around +1475ms for InSeq/OutSeq, +1025ms for Odor
Identity) meaningfully outperforms the fixed 500ms-at-Poke-In window used everywhere before it, on Mitt.
Before adopting that as a new default, or building a full multi-rat pooled training pipeline, this
notebook checks whether the same pattern holds across all 5 rats, or whether it was specific to Mitt.

**Approach:** the exact same sliding-window scan from notebook 008 (250ms window, 25ms steps, -500ms to
+1500ms relative to Poke-In, spectral features with the updated bands), run separately on each of the 5
rats. Rats are kept SEPARATE here, not pooled into one combined dataset, this notebook is about checking
generalization of the window-timing pattern, not yet about combining data for more training examples,
that's a following step once this is reviewed.

**Runtime note:** this trains and evaluates 81 window positions x 2 tasks x 5 rats = a meaningfully
larger computation than notebook 008 alone. Expect this to take several minutes, not seconds, let it run.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.preprocessing import build_labels, get_sampling_rate


## 1. Identify All Sessions

In [ ]:
raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(session_dirs)} sessions:")
for p in session_dirs:
    print(" ", p.name)


## 2. Shared Settings (Same as Notebook 008)

In [ ]:
WINDOW_LENGTH_MS = 250
STEP_MS = 25
SPAN_START_MS = -500
SPAN_END_MS = 1500

offsets_ms = np.arange(SPAN_START_MS, SPAN_END_MS + 1, STEP_MS)

BANDS = {
    'delta': (1, 4),
    'theta': (4, 12),
    'beta': (12, 30),
    'low_gamma': (30, 80),
    'high_gamma': (80, 150),
}

print(f"{len(offsets_ms)} window positions per rat")


## 3. Helper Functions (Same as Notebook 008)

In [ ]:
def extract_window_at_offset(lfp_data, timebin, poke_idx, offset_ms, window_ms, target_samples):
    poke_time = timebin[poke_idx]
    start_time = poke_time + offset_ms / 1000
    end_time = start_time + window_ms / 1000

    start_idx = np.searchsorted(timebin, start_time)
    end_idx = np.searchsorted(timebin, end_time)
    if end_idx - start_idx < 2:
        return None

    raw_window = lfp_data[:, start_idx:end_idx]
    n_channels, n_raw = raw_window.shape
    old_x = np.linspace(0, 1, n_raw)
    new_x = np.linspace(0, 1, target_samples)
    resampled = np.zeros((n_channels, target_samples))
    for ch in range(n_channels):
        resampled[ch] = np.interp(new_x, old_x, raw_window[ch])
    return resampled


def band_power_features(windows, fs, bands):
    n_trials, n_channels, n_samples = windows.shape
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    band_masks = {name: (freqs >= lo) & (freqs < hi) for name, (lo, hi) in bands.items()}
    n_features = n_channels * len(bands)
    X = np.zeros((n_trials, n_features))
    col = 0
    for ch in range(n_channels):
        fft_vals = np.fft.rfft(windows[:, ch, :], axis=1)
        power = np.abs(fft_vals) ** 2
        for band_name, mask in band_masks.items():
            X[:, col] = power[:, mask].sum(axis=1)
            col += 1
    return X


def run_sliding_window_for_session(session_dir, offsets_ms, window_length_ms, bands):
    """Runs the full sliding-window scan for one rat's session, returns both decoding curves."""
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_data = lfp['data']

    timebin = bvr_data[bvr_keys.index('TimeBin')]
    avg_fs = get_sampling_rate(bvr_data, bvr_keys)
    labels = build_labels(bvr_data, bvr_keys)
    target_samples = int(round(window_length_ms / 1000 * avg_fs))

    min_time_needed_before = abs(offsets_ms[0]) / 1000
    max_time_needed_after = (offsets_ms[-1] + window_length_ms) / 1000

    valid_trial_idx = []
    for t in labels['trial_idx']:
        poke_time = timebin[t]
        if poke_time - min_time_needed_before < timebin[0]:
            continue
        if poke_time + max_time_needed_after > timebin[-1]:
            continue
        valid_trial_idx.append(t)
    valid_trial_idx = np.array(valid_trial_idx)
    valid_mask = np.isin(labels['trial_idx'], valid_trial_idx)
    inseq_outseq = labels['inseq_outseq'][valid_mask]
    odor_id = labels['odor_id'][valid_mask]

    inseq_curve, odor_curve = [], []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for offset in offsets_ms:
        windows = []
        for t in valid_trial_idx:
            w = extract_window_at_offset(lfp_data, timebin, t, offset, window_length_ms, target_samples)
            windows.append(w)
        windows = np.stack(windows, axis=0)
        X = np.log1p(band_power_features(windows, avg_fs, bands))

        pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
        inseq_scores = cross_val_score(pipe, X, inseq_outseq, cv=skf, scoring='balanced_accuracy')
        inseq_curve.append(inseq_scores.mean())

        inseq_mask_local = (inseq_outseq == 1)
        X_odor = X[inseq_mask_local]
        y_odor = odor_id[inseq_mask_local]
        skf_odor = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        odor_scores = cross_val_score(pipe, X_odor, y_odor, cv=skf_odor, scoring='balanced_accuracy')
        odor_curve.append(odor_scores.mean())

    return {
        'session': session_name,
        'n_trials': len(valid_trial_idx),
        'inseq_curve': np.array(inseq_curve),
        'odor_curve': np.array(odor_curve),
    }


## 4. Run the Sliding Window Scan for Every Rat

This is the long-running cell in this notebook, it repeats notebook 008's full analysis 5 times, once
per rat. Progress is printed after each rat finishes.


In [ ]:
all_results = {}
for session_dir in session_dirs:
    print(f"Running {session_dir.name}...")
    result = run_sliding_window_for_session(session_dir, offsets_ms, WINDOW_LENGTH_MS, BANDS)
    all_results[session_dir.name] = result
    print(f"  done, {result['n_trials']} trials used, "
          f"InSeq/OutSeq best={result['inseq_curve'].max():.3f}, "
          f"Odor best={result['odor_curve'].max():.3f}")

print("\nAll rats complete.")


## 5. Plot All 5 Rats Together

Two plots: InSeq/OutSeq decoding curves for all rats overlaid, then the same for Odor Identity. If the
"later window is better" pattern from Mitt is real and general, we should see multiple rats' curves
trending upward toward the later offsets, not just Mitt.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

colors = plt.cm.tab10(np.linspace(0, 1, len(all_results)))

for (session_name, result), color in zip(all_results.items(), colors):
    axes[0].plot(offsets_ms, result['inseq_curve'], label=session_name, color=color, linewidth=1.8)
axes[0].axhline(0.5, color='black', linestyle='--', alpha=0.4, label='chance (0.5)')
axes[0].axvline(0, color='black', linestyle=':', alpha=0.6)
axes[0].set_xlabel('Window start time relative to Poke-In (ms)')
axes[0].set_ylabel('Balanced accuracy')
axes[0].set_title('InSeq/OutSeq decoding curve, all rats')
axes[0].legend(fontsize=8, loc='upper left')

for (session_name, result), color in zip(all_results.items(), colors):
    axes[1].plot(offsets_ms, result['odor_curve'], label=session_name, color=color, linewidth=1.8)
axes[1].axhline(0.2, color='black', linestyle='--', alpha=0.4, label='chance (0.2)')
axes[1].axvline(0, color='black', linestyle=':', alpha=0.6)
axes[1].set_xlabel('Window start time relative to Poke-In (ms)')
axes[1].set_ylabel('Balanced accuracy')
axes[1].set_title('Odor Identity decoding curve, all rats')
axes[1].legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()


## 6. Per-Rat Summary Table

Best offset and best balanced accuracy per rat, per task, plus the hypothesis check (does odor peak
before InSeq/OutSeq) evaluated separately for every rat, not just Mitt.


In [ ]:
print(f"{'Rat':20s} {'InSeq best':12s} {'@ offset':10s} {'Odor best':12s} {'@ offset':10s} {'Odor peaks first?':18s}")
print("-" * 85)

summary_rows = []
for session_name, result in all_results.items():
    inseq_best_idx = result['inseq_curve'].argmax()
    odor_best_idx = result['odor_curve'].argmax()
    inseq_best = result['inseq_curve'][inseq_best_idx]
    odor_best = result['odor_curve'][odor_best_idx]
    inseq_offset = offsets_ms[inseq_best_idx]
    odor_offset = offsets_ms[odor_best_idx]
    odor_first = odor_offset < inseq_offset

    print(f"{session_name:20s} {inseq_best:<12.3f} {inseq_offset:<10d} {odor_best:<12.3f} {odor_offset:<10d} {str(odor_first):18s}")

    summary_rows.append({
        'session': session_name,
        'inseq_best_balanced_accuracy': round(float(inseq_best), 4),
        'inseq_best_offset_ms': int(inseq_offset),
        'odor_best_balanced_accuracy': round(float(odor_best), 4),
        'odor_best_offset_ms': int(odor_offset),
        'odor_peaks_before_inseq': bool(odor_first),
    })


## 7. Text-Only Results Export


In [ ]:
import json as _json
import os

results_summary = {
    "purpose": "Multi-rat validation of notebook 008's sliding-window finding",
    "window_length_ms": WINDOW_LENGTH_MS,
    "step_ms": STEP_MS,
    "span_ms": [int(SPAN_START_MS), int(SPAN_END_MS)],
    "bands": {name: list(rng) for name, rng in BANDS.items()},
    "per_rat_summary": summary_rows,
    "n_rats_where_odor_peaks_first": sum(row['odor_peaks_before_inseq'] for row in summary_rows),
    "n_rats_total": len(summary_rows),
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook009_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook009_results.json")


## 8. Written Summary Report

**Objective**

Check whether the sliding-window finding from notebook 008 (later windows outperform the fixed 500ms
window; Odor Identity peaks before InSeq/OutSeq) generalizes across all 5 rats, or was specific to Mitt.

**Method**

The exact sliding-window scan from notebook 008 (250ms window, 25ms steps, -500 to +1500ms relative to
Poke-In, spectral features with updated bands) repeated independently on each of the 5 rats. Rats were
NOT pooled into one dataset, each rat's curves were computed entirely separately.

**Results**

*(Fill in after reviewing Section 6's table and Section 5's plots.)* How many of the 5 rats show the
"odor peaks before InSeq/OutSeq" pattern? Do the curves broadly agree in shape across rats, or does each
rat look meaningfully different?

**Interpretation**

*(Fill in.)* If most/all rats agree: the window-timing pattern is likely a real, general property of
this task, worth adopting as a new default window going forward. If rats disagree substantially: either
per-rat window tuning is needed (more complex, but possibly still worthwhile), or the "best offset" from
any single rat is more noise-driven than real, and a fixed, moderate window (not chasing each rat's
individual peak) may be the more defensible choice for a pooled model.

**Next Steps**

1. Based on the level of agreement found here, decide: adopt one shared "later" window for all rats, or
   proceed with per-rat window tuning.
2. Move to building the actual multi-rat pooled training pipeline (session-based train/test split, not
   trial-level), using whichever window decision this analysis supports.
3. Once pooled results exist, run a permutation test to confirm significance before presenting a final
   number.
